# `stmux` — source code

The three files that make up `stmux` on Isambard-AI, verbatim.

| file | live path | role |
|---|---|---|
| `bashrc_stmux_section.sh` | `/lus/lfs1aip2/projects/public/u6gb/.bashrc` lines 76-299 | the `stmux` shell function and the cross-node breadcrumb |
| `stmux-tmux` | `/lus/lfs1aip2/projects/public/u6gb/.local/bin/stmux-tmux` | helper: `--diagnose` / `--record` / `--kill-stale` / `--sweep-deleted` |
| `stmux-on-session-start.sh` | `/lus/lfs1aip2/projects/public/u6gb/.claude/hooks/stmux-on-session-start.sh` | Claude SessionStart hook, creates `claude-<node>` |


In [1]:
from pathlib import Path
from IPython.display import Code, display
SRC = Path('src')
def show(name):
    p = SRC / name
    print(f'{p}  —  {len(p.read_text().splitlines())} lines, {p.stat().st_size} bytes')
    display(Code(str(p), language='bash'))


## 1. `~/.bashrc` lines 76-299 — the `stmux` function

In [2]:
show('bashrc_stmux_section.sh')

src/bashrc_stmux_section.sh  —  224 lines, 14630 bytes


# >>> stmux: module load brics tmux + 新建/接回 session (2026-08-11) >>>
# stmux <name>  =  module load brics/tmux/3.4 && tmux new -A -s <name>
#   -A: session 已存在就 attach 回去,不存在才新建(否则第二次敲会 duplicate session 报错)
# stmux         =  自动命名 stmux-<UTC时间戳> 并进入(2026-08-12 改,原为列表)
# stmux -l|ls   =  跨节点 breadcrumb 表 + 本节点 tmux ls
#
# 2026-08-12 无参数改为"自动起名新建"。动机: 起名是打断,大多数时候只是想要一张新的
#   工作区。时间戳取 UTC 是因为登录节点时区本身就是 UTC(date 与 date -u 输出一致),
#   不存在本地/UTC 错位。副作用要知道: 时间戳每秒不同 => 无参数每次都是新 session,
#   永远不会接回旧的; 要接回必须显式给名字。原列表功能整块挪到 -l/--list/ls。
#
# 2026-08-12 加 breadcrumb。动机(实测于当日): tmux server 是登录节点本地的,socket 在
#   /tmp/tmux-<uid>/ 而 /tmp 不跨节点;`ssh isambard` 走负载均衡,每次随机落到
#   login01/02/03/42/44/45 之一;登录节点之间 port 22 不通(2026-06-24 实测),无法跨
#   节点代查进程或 tmux。后果:落错节点时 `tmux ls` 返回空,看起来 session 死了,其实
#   活在别的节点上 —— 2026-08-11 的 hybrid 就是这样,它建在 login42,重连落到 login44。
#   解法:HOME=/projects/public/u6gb 在 Lustre 上跨节点共享,把 session→节点映射写成
#   一个小文件。这是 CLAUDE.md breadcrumb 原则的直接应用:不靠探测,靠写下来的事实。
_STMUX_BC="$HOME/.tmux-sessions.tsv"

# 2026-08-14 钉死 tmux 二进制。动机(当日实测): 一个 socket 只能有一个 tmux 二进制。
#   SessionStart hook 用绝对路径的 miniforge tmux 3.6 在 /tmp/tmux-<uid>/default 上
#   建了 server,而下面 stmux() 第一行 `module load brics/tmux/3.4` 把 3.4 顶到 PATH
#   最前 —— 3.4 的 client 连 3.6 的 server 握手对不上,报 "server exited unexpectedly"。
#   这句话是骗人的: server 一直活着(用 3.6 的客户端 `tmux ls` 当场能看到 session)。
#   误导性在于它把"协议不兼容"说成了"对方死了",于是人会去重建 session,而重建会
#   把还在里面跑的东西(当天是 fx488 恢复驱动)一起带走。
#   解法: tmux 一律走一个绝对路径。选 miniforge 那个,因为它不需要 module 就在默认
#   PATH 上,也正是 hook 用的那个。module load 一行保留(它还负责别的环境变量),
#   只是不再决定"用哪个 tmux"。
#   逃生口 —— 要接回历史上由 3.4 建的 server(别的登录节点可能还有):
#       _STMUX_TMUX=$(module load brics/tmux/3.4; command -v tmux) stmux <name>
#
# 2026-09-03 上面那条修复只修了三条路径中的一条,所以同一个报错又出现了(login44)。
#   一个 socket 上有三条路径各自独立地决定用哪个 tmux:
#       stmux()            _STMUX_TMUX 钉死 3.6 —— 但底下那句 `|| _STMUX_TMUX=tmux`
#                          在钉死的路径取不到时**静默退回 PATH**,而 PATH 第一位正是
#                          上一行 module load 刚顶上去的 3.4。那不是安全网,是上膛。
#       SessionStart hook  `command -v tmux`,完全看 PATH,根本没钉
#       用户随手敲 tmux    module load 之后就是 3.4
#   实测方向是单向的: 3.6 client 连 3.4 server rc=0,反过来 rc=1 报那句谎话。
#   所以修法是把"用哪个 tmux"收拢成单一事实来源,三条路径都去问同一个脚本:
_STMUX_HELPER="$HOME/.local/bin/stmux-tmux"
# 2026-09-03(下午,用户令) tmux 二进制写死成一个绝对路径,不再从帮助脚本推导。
#   在此之前这个变量有**两个**赋值点,只有一个做了校验:
#       这里          _STMUX_TMUX=$(帮助脚本),后面跟 ${...:-固定路径} 兜底 —— 保证非空
#       stmux() 里    _STMUX_TMUX=$(帮助脚本) || return 1        —— 没有兜底
#   当日实测的后果:socket 是陈旧文件时帮助脚本静默 exit 1 且不输出任何东西,
#   函数里那句便把这个**全局**变量写成空(赋值先于 || 完成,|| 拦不住它;且函数里
#   没有 local)。此后本终端每次 stmux 都拿空串当命令跑 -> rc=127 -> 被读成
#   "连不上 server",而同一屏的诊断表显示 OK、kill-stale 说"连得上不是 stale"
#   —— 三句话自相矛盾,真正错的只有读了空变量的那一句,但正是它把流程带进死路。
#   固定路径没有这个失败模式:它不依赖任何运行时状态,也就没有"取不到"这一说。
#   /tools/brics/... 是 root 所有、全站 1000+ 用户共享的那份,不会被删或改名。
#   帮助脚本保留,但只做三件不决定"用哪个 tmux"的事:--diagnose / --record / --kill-stale。
_STMUX_TMUX=/tools/brics/apps/linux-sles15-neoverse_v2/gcc-12.3.0/tmux-3.4-5vcftkte724cekyuashr2ex65c5fpfxj/bin/tmux
# 取不到就报错,不退回 PATH 上的 `tmux`: 那正是当初撞版本的来路,而报错信息谎称
# server 死了,人会去重建 session,把里面在跑的东西一起带走。宁可这里不动。
if [ ! -x "$_STMUX_TMUX" ]; then
    echo "stmux: 找不到可用的 tmux ($_STMUX_TMUX)。诊断: $_STMUX_HELPER --diagnose" >&2
fi

# 让裸敲的 `tmux` 也拿到同一个 3.4。3.6 改名之后 PATH 上一个 tmux 都没有了,
# 而 3.4 只存在于 module 树里 —— 这一句是它进 PATH 的唯一途径。
# 注意这与"stmux 函数里删掉 module load"不矛盾: 当时删是因为它引入了**第二个**
# 版本;现在 3.6 已不在 PATH 上,这一句带进来的正是唯一的那个,裸敲不再有歧义。
module load brics/tmux/3.4 >/dev/null 2>&1

# 记一行 <节点>TAB<session>TAB<UTC>;同 (节点,session) 覆盖。写临时文件再 mv,原子替换。
_stmux_bc_record() {
    local _n _t _tmp
    _n=$(hostname); _t=$(date -u +%Y-%m-%dT%H:%M:%SZ)
    _tmp=$(mktemp "${_STMUX_BC}.XXXXXX" 2>/dev/null) || return 0
    {
        [ -f "$_STMUX_BC" ] && awk -F'\t' -v n="$_n" -v s="$1" '!($1==n && $2==s)' "$_STMUX_BC"
        printf '%s\t%s\t%s\n' "$_n" "$1" "$_t"
    } > "$_tmp" 2>/dev/null
    mv -f "$_tmp" "$_STMUX_BC" 2>/dev/null || rm -f "$_tmp"
}

# 打印全表。本节点的行用 has-session 当场验证并顺手清理死记录(GC 只发生在验证得了的
# 地方);别节点的行一律

## 2. `~/.local/bin/stmux-tmux` — single source of truth for the tmux binary

In [3]:
show('stmux-tmux')

src/stmux-tmux  —  245 lines, 12691 bytes


#!/bin/bash
# stmux-tmux — 回答"这台机器上该用哪个 tmux 二进制",作为单一事实来源。
#
# 2026-09-03。起因:login44 上 `stmux` 报 "server exited unexpectedly"。
#
# 机制:tmux 的 socket 是 per-uid 的(/tmp/tmux-<uid>/default),一台登录节点上你所有
# 的 tmux 共用它 —— 谁先建 server,谁就定死了这个 socket 的协议版本。当时机器上同时
# 存在两个 tmux:
#     3.6  miniforge base(conda 包 tmux-3.6-h2fb902b_0),默认在 PATH 上
#     3.4  /tools/brics/...(module brics/tmux/3.4),root 所有,全站 1000+ 用户共享
#
# 实测(2026-09-03,login45,真实 socket):
#     3.6 client → 3.4 server   rc=0   能连
#     3.4 client → 3.6 server   rc=1   "server exited unexpectedly"
# 兼容单向:新客户端认旧 server,旧客户端不认新 server。那句报错是谎话 —— server 一直
# 活着,换 3.6 的客户端当场就能列出 session。危害在于它把"协议不兼容"说成"对方死了",
# 于是人会去重建 session,把里面在跑的东西一起带走。
#
# 已按用户令删掉 3.6(二进制 + conda 包记录一起清 —— 只删文件的话下次 conda 操作会
# 把它装回来,而那时已经没人记得为什么)。现在整台机器上只有 3.4,冲突从源头消失。
#
# 那这个脚本还有什么用?三件事,"只留一个"保证不了:
#   1. 3.4 只活在 module 树里。非交互 shell(hook、cron、srun)没有 Lmod 注入的
#      module function,`command -v tmux` 会是空 —— 用绝对路径才稳。
#   2. 哪天再冒出第二个 tmux(conda 装回来、系统换了 module、别人给你 source 了
#      什么),--diagnose 能当场说清"谁连得上这个 socket"。
#   3. 记账。"这个 socket 上的 server 当初是谁建的"只存在于那个跑着的进程里,
#      ls / cat / 读脚本都看不到 —— 所以建完就写进 ~/.tmux-servers/<节点>.tsv。
#      当所有已知客户端都连不上时,探测只能说"都连不上",而记录能说出那个二进制
#      是什么。这是 breadcrumb 原则的第二次应用(第一次是 .tmux-sessions.tsv 解决
#      "session 在哪台节点")。
#
# 用法:
#     stmux-tmux                打印该用的 tmux 绝对路径(拿不到则 rc=1,无输出)
#     stmux-tmux --diagnose     socket 现状、候选连通性、本节点的建立记录
#     stmux-tmux --record <谁>  刚建完 server 时调用,记一行账
#     stmux-tmux --kill-stale   socket 被一个谁都连不上的 server 占着时,清掉它
#                               (加 --yes 跳过确认)

_PRIMARY=(
    /tools/brics/apps/linux-sles15-neoverse_v2/gcc-12.3.0/tmux-3.4-5vcftkte724cekyuashr2ex65c5fpfxj/bin/tmux
)

_LOGDIR="${HOME}/.tmux-servers"
_LOG="${_LOGDIR}/$(hostname).tsv"

_socket() { echo "${TMUX_TMPDIR:-/tmp}/tmux-$(id -u)/default"; }
_ver()    { local v; v=$("$1" -V 2>/dev/null) || return 1; echo "${v#tmux }"; }

# 当前 socket 上 server 的 pid(用能连上的客户端问它自己)
_server_pid() { "$1" display-message -p '#{pid}' 2>/dev/null; }

# 记录里对应"当前这个 server"的那一行 -> 打印它记的二进制路径。
# 判断依据是 server_pid 还活着且确实是个 tmux —— pid 会被复用,所以两条都要查。
_binary_from_log() {
    [ -r "$_LOG" ] || return 1
    local ts pid ver bin who exe
    while IFS=$'\t' read -r ts pid ver bin who; do
        case "$ts" in ''|'#'*) continue ;; esac
        [ -n "$pid" ] && [ -d "/proc/$pid" ] || continue
        exe=$(readlink -f "/proc/$pid/exe" 2>/dev/null) || continue
        case "$exe" in *tmux*) ;; *) continue ;; esac
        echo "$bin"
    done < "$_LOG" | tail -1     # 同一 pid 可能记过多次,取最后一行
}

# 占着这个 socket 的 server 进程。ss 把 unix socket 路径直接关联到 pid,
# 比"exe 含 tmux + ppid==1"准 —— 后者会把别的 socket(tmux -L xxx)的 server
# 一起捞进来,误杀。
_server_procs() {
    ss -xlp 2>/dev/null | grep -F "$(_socket) " \
        | grep -oE 'pid=[0-9]+' | cut -d= -f2 | sort -u
}

# ── --sweep-deleted ────────────────────────────────────────────────────────────
# 2026-09-03(下午,用户令"kill 3.6 keep 3.4") 无条件清掉**二进制已被删除**的 tmux server。
#
# 为什么这条规则安全,而"杀掉连不上的 server"不安全:
#   连不上有两种成因 —— 协议版本对不上(客户端还在,换个客户端就能无损接回)、
#   以及二进制被删了(没有任何客户端能被造出来)。前者杀了是浪费,后者不杀只是让
#   socket 一直被占着。readlink /proc/<pid>/exe 结尾的 " (deleted)" 精确区分这两者。
#
# 本环境的实例:miniforge 的 tmux 3.6 于 2026-09-03 16:46 被删,而它 16:07 在多个
#   登录节点上建的 server 还活着。只剩 3.4 客户端,而 3.4 连 3.6 是单向不兼容的
#   ("server exited unexpectedly")。恢复那个二进制的路全断了 ——
#   /proc/<pid>/exe 在 Lustre 上取回是 ESTALE(删了就没了,不像本地盘)、
#   conda pkgs 缓存已清、/usr/bin/tmux 不存在。所以杀是唯一出路。
#
# 登录节点之间 port 22 不通(2026-09-03 实测 5 个节点全 timeout),没法从一台扫全部,
# 所以这件事必须在**每次落到一个节点时自动发生** —— stmux() 和 SessionStart hook
# 都会调它。不问 y/N:被删掉的二进制没有"再想想"的余地。
if [ "${1-}" = "--sweep-deleted" ]; then
    _dir=$(dirname "$(_socket)")
    [ -d "$_dir" ] || exit 0
    _n=0
    for _s in "$_dir"/*; do
        [ -S "$_s" ] || continue
        for _p in $(ss -xlp 2>/dev/null | grep -F "$_s " | grep -oE 'pid=[0-9]+' | cut -d= -f2 | sort -u); do
            _exe=$(readlink "/proc/$_p/exe" 2>/dev/null) || continue
            case "$_exe" in *tmux*) ;; *) continue ;; 

## 3. SessionStart hook — `claude-<node>` workspace

In [4]:
show('stmux-on-session-start.sh')

src/stmux-on-session-start.sh  —  99 lines, 6430 bytes


#!/bin/bash
# SessionStart hook (2026-08-14, 用户要求"每次登录 Isambard 都敲 stmux")
#
# 保证当前节点上存在一张属于 Claude 的 tmux session,并把它的名字告诉 Claude。
#
# 为什么不是直接调 `stmux`:
#   1. stmux 是 ~/.bashrc 里的 shell function,不是可执行文件 —— hook 的非交互
#      子 shell 里这个名字根本不存在;它依赖的 `module` 也是 Lmod 注入的 function。
#   2. stmux 无参走 `tmux new -A`,即 attach 分支。Claude 的 Bash 工具没有 tty
#      (`tty` → not a tty),attach 必报 "open terminal failed: not a terminal"。
#   3. stmux 无参用 UTC 时间戳命名,每次都是新 session。SessionStart 在
#      startup/resume/clear/compact 都会触发,照搬会堆出一地孤儿 session。
#      这里改用按节点固定的名字,所以是"建或复用",重连能接回原来那张。
#
# 失败一律 exit 0:这个 hook 绝不允许拖住/挡住会话启动。

_STMUX_BC="${HOME}/.tmux-sessions.tsv"

# ── 1. 砍掉继承来的 $TMUX ────────────────────────────────────────────────
# $TMUX 是 socket 路径,不是"我在不在 tmux 里"的布尔量。Claude 经常跑在计算节点上
# (srun/sbash 进去的),而 $TMUX 是从登录节点 shell 继承来的,它指向的 socket 在
# /tmp 里 —— 而 /tmp 是节点本地的,在计算节点上不存在。留着它,tmux 会照那个路径
# 在计算节点上另起一台幽灵 server,`tmux ls` 默认 socket 完全看不见,现象酷似
# "session 没建起来"。2026-08-12 的 `fake` socket 事故就是这个。
unset TMUX TMUX_PANE

# ── 2. 找 tmux 二进制 ───────────────────────────────────────────────────
# 2026-09-03 改:原来这里是 `command -v tmux`,即"PATH 上的第一个"。那是个 bug。
#   tmux 的 socket 是 per-uid 的(/tmp/tmux-<uid>/default),这台机器上你所有的 tmux
#   共用它 —— 谁先建 server,谁就定死了这个 socket 的协议版本。而本环境里有两个
#   tmux(miniforge 3.6 / module brics 3.4),兼容是单向的:3.6 client 连 3.4 server
#   可以,3.4 client 连 3.6 server 报 "server exited unexpectedly"。那句话是谎话,
#   server 活着,只是握手对不上 —— 但它会把人骗去重建 session,连带杀掉里面在跑的东西。
#   看 PATH 就意味着:这个 hook 用哪个版本建 server,取决于启动 claude 时那个 shell
#   有没有 module load 过。同一个 socket 上,hook / stmux / 用户裸敲的 tmux 三条路径
#   各自独立决定版本,只钉死其中一条(2026-08-14 只钉了 stmux)修不好。
#   现在三条都问同一个脚本。
_STMUX_HELPER="${HOME}/.local/bin/stmux-tmux"
[ -x "$_STMUX_HELPER" ] || _STMUX_HELPER=/projects/public/u6gb/.local/bin/stmux-tmux
# 2026-09-03(下午,用户令) 先清掉二进制已被删除的 tmux server,再去找该用哪个。
#   顺序不能反:那种 server 占着 socket,任何客户端都连不上,下面的 new-session 必失败,
#   而 hook 是 exit 0 静默的 —— 会表现为"工作区莫名其妙没建起来"。
#   stderr 丢掉:hook 的输出会进 Claude 的上下文,清理是例行动作不值得每次刷屏。
[ -x "$_STMUX_HELPER" ] && "$_STMUX_HELPER" --sweep-deleted 2>/dev/null

_tmux=""
[ -x "$_STMUX_HELPER" ] && _tmux=$("$_STMUX_HELPER" 2>/dev/null)
if [ -z "$_tmux" ] || [ ! -x "$_tmux" ]; then
    # helper 不在(别的机器/别的账号)才退回原来的找法
    _tmux=$(command -v tmux 2>/dev/null)
    if [ -z "$_tmux" ]; then
        # module 是 Lmod 注入的 shell function,非交互子 shell 里没有,得先 source init
        [ -r /opt/cray/pe/lmod/lmod/init/bash ] && . /opt/cray/pe/lmod/lmod/init/bash >/dev/null 2>&1
        module load brics/tmux/3.4 >/dev/null 2>&1
        _tmux=$(command -v tmux 2>/dev/null)
    fi
fi
[ -n "$_tmux" ] || exit 0

# ── 3. 建或复用 ─────────────────────────────────────────────────────────
_node=$(hostname 2>/dev/null) || exit 0
_name="claude-${_node}"

# 不用 `new-session -A -d`:-A 命中已存在的 session 时会走 attach 分支,-d 拦不住
# (man 只用一句从句写"-A 时 -D 才 behave like -d"),无 tty 下第二次调用直接 rc=1。
# has-session 的 -t 必须加 "=" 前缀强制精确匹配,否则前缀/fnmatch 匹配会误命中。
_state=reused
if ! "$_tmux" has-session -t "=${_name}" 2>/dev/null; then
    "$_tmux" new-session -d -s "${_name}" -c "${CLAUDE_PROJECT_DIR:-$HOME}" >/dev/null 2>&1 || exit 0
    _state=created
fi

# 记一笔"这个 socket 上的 server 是谁建的" —— 见 ~/.tmux-servers/README。
# 这件事只存在于跑着的进程里,ls/cat/读脚本都看不到,所以建完就写下来。
[ -x "$_STMUX_HELPER" ] && "$_STMUX_HELPER" --record hook 2>/dev/null

# ── 4. 登记 breadcrumb,让用户的 `stmux -l` 看得见这张 session ───────────
# 格式与 .bashrc 里的 _stmux_bc_record 完全一致:<节点>TAB<session>TAB<UTC>,
# 同 (节点,session) 覆盖,写临时文件再 mv 做原子替换。
_t=$(date -u +%Y-%m-%dT%H:%M:%SZ)
_tmp="${_STMUX_BC}.$$"
{
    [ -f "$_STMUX_BC" ] && awk -F'\t' -v n="$_node" -v s="$_name" '!($1==n && $2==s)' "$_STMUX_BC"
    printf '%s\t%s\t%s\n' "$_node" "$_name" "$_t"
} > "$_tmp" 2>/dev/null
mv -f "$_tmp" "$_STMUX_BC" 2>/dev/null

# ── 5. 把结果交回 Claude 的上下文 ───────────────────────────────────────
cat <<EOF
[stmux] 本节点 ${_node} 上的 Claude tmux 工作区: ${_name} (${_state})
凡是"必须活过本会话"的进程都放进去跑,比 setsid nohup 多一样:输出还能回看。
  ${_tmux} send-keys   -t '=${_name}:' '<命令>' Enter
  ${_tmux} c